# Demo: varying interclonal distance

Two horizontal tumor stripes with increasing center-to-center gap
$d \in \{0, 0.25, 0.5, 0.75, 1\}\cdot d_{\max}$
(spot spacing = 1).

## Requirements
`numpy`, `pandas`, `matplotlib`

## Paths (repository root)
| Role | Path |
|------|------|
| **Input** (optional) | `data/cell_anno.tsv` |
| **Output** | `output/vary_distance/<condition>/` |

### Input file
Same as `01_vary_shape.ipynb`: header-free `barcode<TAB>clone_label` TSV under `data/cell_anno.tsv`.

### Output files (per condition)
| File | Format |
|------|--------|
| `tissue_positions_list.csv` | no header; `barcode, in_tissue, x, y, pixel_row, pixel_col` |
| `spot_anno_pattern.tsv` | TSV with header; barcode index + `spot_anno` |
| `barcodes.tsv.gz` | one barcode per line |

Conditions: `dist_0`, `9.0`, `18.1`, `27.1`, `36.1`.

## Run
```bash
jupyter nbconvert --to notebook --execute 02_vary_distance.ipynb
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

# Run this notebook from the repository root (directory that contains this file).
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "_spatial_pattern_utils.py").exists():
    raise FileNotFoundError(
        "Cannot find _spatial_pattern_utils.py in cwd. "
        "cd to the demo repository root before running."
    )
sys.path.insert(0, str(REPO_ROOT))

from _spatial_pattern_utils import (
    DEFAULT_CELL_ANNO,
    DEFAULT_OUTPUT_DIR,
    SEED,
    TARGET_PER_TYPE,
    load_or_make_barcodes_by_type,
    make_hex_grid,
    map_barcodes_to_labels,
    plot_spatial,
    resolve_repo_root,
    distance_schedule,
    pattern_vary_distance,
    write_pattern_dir,
)

REPO_ROOT = resolve_repo_root()
# Optional input: header-free TSV with columns barcode, clone_label
CELL_ANNO = REPO_ROOT / DEFAULT_CELL_ANNO

OUT_ROOT = REPO_ROOT / DEFAULT_OUTPUT_DIR / "vary_distance"

grid = make_hex_grid()
distances = distance_schedule(grid, n=5)
barcodes_by_type = load_or_make_barcodes_by_type(CELL_ANNO, seed=SEED)
print("REPO_ROOT:", REPO_ROOT)
print("Distances:", [round(float(d), 1) for d in distances])
print("Clone sizes:", TARGET_PER_TYPE)
print("Input cell_anno:", CELL_ANNO if CELL_ANNO.exists() else "(missing → synthetic barcodes)")
print("Output root:", OUT_ROOT)


In [ ]:
def distance_display_name(d: float, i: int) -> str:
    if i == 0 or abs(d) < 1e-9:
        return "dist_0"
    return f"{d:.1f}"

pattern_dirs = {}
for i, d in enumerate(distances):
    labels = pattern_vary_distance(grid, float(d))
    display = distance_display_name(float(d), i)
    barcodes = map_barcodes_to_labels(labels, barcodes_by_type, seed=SEED + 1000 + i)
    outdir = write_pattern_dir(OUT_ROOT / display, grid, labels, barcodes)
    pattern_dirs[display] = (outdir, labels, float(d))
    print(f"{display:10s}  d={d:.3f}  -> {outdir}")


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (display, (_, labels, d)) in zip(axes, pattern_dirs.items()):
    plot_spatial(ax, grid, labels, title=f"{display}\n(d={d:.1f})")
axes[0].legend(loc="upper left", fontsize=7, markerscale=2)
fig.suptitle("Varying interclonal distance", fontsize=12)
fig.tight_layout()
OUT_ROOT.mkdir(parents=True, exist_ok=True)
fig_path = OUT_ROOT / "vary_distance_overview.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved overview figure:", fig_path)
plt.show()
